# Projeto Final — DS-PY-004  
## Análise Exploratória de Dados Eleitorais (TSE) 

Este notebook apresenta uma análise exploratória dos dados de votação por seção eleitoral dos estados **MG**, **RJ** e **SC**, utilizando a base oficial do **TSE**. O foco é entender padrões de abstenção, votos brancos, nulos e válidos, além de investigar diferenças entre municípios, zonas eleitorais e possíveis outliers.

---

## 1. Apresentação da base e perguntas de análise

### 1.1 Origem da base

- Fonte: Dados abertos do **Tribunal Superior Eleitoral (TSE)**  
- Recorte: Eleições gerais estaduais de **2022**, primeiro turno  
- Unidade de análise: **Seção eleitoral**  
- Estados analisados: **Minas Gerais (MG)**, **Rio de Janeiro (RJ)** e **Santa Catarina (SC)**  

Cada linha representa uma seção eleitoral, com informações sobre:

- município, zona e seção  
- número de eleitores aptos  
- comparecimento, abstenções  
- votos brancos, nulos, nominais e de legenda  
- cargo, turno, modelo de urna, entre outros.

### 1.2 Perguntas que queremos responder

1. **Abstenção**:  
   Há relação entre o tamanho da seção (eleitores aptos) e a taxa de abstenção?

2. **Brancos e nulos**:  
   Municípios com maior abstenção também têm maior proporção de votos brancos e nulos?

3. **Diferenças internas**:  
   Há diferenças relevantes entre zonas eleitorais dentro de um mesmo município?

4. **Outliers**:  
   A distribuição de votos válidos por seção apresenta outliers? Eles são erros ou refletem seções específicas?

5. **Comparação entre estados**:  
   RJ, MG e SC apresentam padrões distintos de votos válidos e abstenção?

---

## 2. Importação das bibliotecas e leitura dos dados


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

df_rj = pd.read_csv(
    "C:/Users/leona/Downloads/TEC_PROGAMACAO/Grupo_7/dados/detalhe_votacao_secao_2022_RJ.csv",
    encoding="latin1",
    sep=";",
)
df_mg = pd.read_csv(
    "C:/Users/leona/Downloads/TEC_PROGAMACAO/Grupo_7/dados/detalhe_votacao_secao_2022_MG.csv",
    encoding="latin1",
    sep=";",
)
df_sc = pd.read_csv(
    "C:/Users/leona/Downloads/TEC_PROGAMACAO/Grupo_7/dados/detalhe_votacao_secao_2022_SC.csv",
    encoding="latin1",
    sep=";",
)
df_rj["estado"] = "RJ"
df_mg["estado"] = "MG"
df_sc["estado"] = "SC"

df = pd.concat([df_rj, df_mg, df_sc], ignore_index=True)
df.info()

# 3 Diagnóstico de qualidade (faltantes, duplicados, categorias, inválidos, outliers)

In [ ]:
# Faltantes
faltantes = df.isna().sum()
faltantes_percent = df.isna().mean() * 100

print(faltantes)
print(faltantes_percent)


In [ ]:
# Duplicados por chave composta
duplicados = df.duplicated(
    subset=["estado", "NM_MUNICIPIO", "NR_ZONA", "NR_SECAO", "CD_CARGO", "NR_TURNO"]
).sum()
duplicados


In [ ]:
# Padronização de município
df["NM_MUNICIPIO_PAD"] = df["NM_MUNICIPIO"].str.strip().str.upper()
inconsistencias_municipio = (df["NM_MUNICIPIO"] != df["NM_MUNICIPIO_PAD"]).sum()
inconsistencias_municipio


In [ ]:
# Valores inválidos
invalid_aptos = (df["QT_APTOS"] < 0).sum()
invalid_comparecimento = (df["QT_COMPARECIMENTO"] < 0).sum()
invalid_brancos = (df["QT_VOTOS_BRANCOS"] < 0).sum()
invalid_nulos = (df["QT_VOTOS_NULOS"] < 0).sum()
comparecimento_maior_aptos = (df["QT_COMPARECIMENTO"] > df["QT_APTOS"]).sum()

invalid_aptos, invalid_comparecimento, invalid_brancos, invalid_nulos, comparecimento_maior_aptos


In [ ]:
# Criar votos válidos no df geral (necessário para outliers)
df["QT_VOTOS_VALIDOS"] = df["QT_VOTOS_NOMINAIS"] + df["QT_VOTOS_LEGENDA"]


##  3.1 — Outliers por Estado (IQR) + Impacto na Mediana

In [ ]:
# Garantir que votos válidos existem no df geral
df["QT_VOTOS_VALIDOS"] = df["QT_VOTOS_NOMINAIS"] + df["QT_VOTOS_LEGENDA"]

outliers_estado = {}
mediana_original = {}
mediana_sem_outliers = {}

for est in df["estado"].unique():
    
    grupo = df[df["estado"] == est]["QT_VOTOS_VALIDOS"]
    
    # Mediana original
    mediana_original[est] = grupo.median()
    
    # IQR
    Q1 = grupo.quantile(0.25)
    Q3 = grupo.quantile(0.75)
    IQR = Q3 - Q1
    
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    
    # Outliers
    mask_outliers = (grupo < lim_inf) | (grupo > lim_sup)
    outliers_estado[est] = mask_outliers.sum()
    
    # Mediana sem outliers
    mediana_sem_outliers[est] = grupo[~mask_outliers].median()

outliers_estado, mediana_original, mediana_sem_outliers


In [ ]:
#Gráfico — Impacto dos Outliers na 

dados = pd.DataFrame(
    {
        "Estado": list(mediana_original.keys()),
        "Mediana Original": list(mediana_original.values()),
        "Mediana Sem Outliers": list(mediana_sem_outliers.values()),
    }
)


dados_longo = dados.melt(
    id_vars=["Estado"],
    value_vars=["Mediana Original", "Mediana Sem Outliers"],
    var_name="Tipo de Mediana",
    value_name="Mediana de Votos Válidos",
)


fig = px.line(
    dados_longo,
    x="Estado",
    y="Mediana de Votos Válidos",
    color="Tipo de Mediana",
    markers=True,  # Adiciona os pontos nos vértices
    title="<b>Impacto dos Outliers na Mediana de Votos Válidos por Estado</b>",
    color_discrete_map={
        "Mediana Original": "#1f77b4",  # Azul
        "Mediana Sem Outliers": "#ff7f0e",  # Laranja
    },
)


fig.update_traces(marker=dict(size=8))  # Aumenta o tamanho dos pontos

fig.update_layout(
    template="plotly_white",
    legend_title_text="",  # Remove o título da legenda
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),  # Legenda no topo
    hovermode="x unified",  # Compara os dois valores ao passar o mouse sobre o estado
)

fig.show()


#### Análise do 3.1 Outliers por Estado (IQR)

Aplicamos o método IQR para identificar outliers na variável `QT_VOTOS_VALIDOS` dentro de cada estado.  
Os resultados foram:

- **RJ:** 2.031 outliers  
- **MG:** 5.126 outliers  
- **SC:** 2.462 outliers  

Esses valores não representam erros — são características estruturais das seções eleitorais:

- **MG** possui muitos municípios pequenos, com seções pequenas e grande variabilidade → mais outliers.  
- **RJ** possui seções grandes e pequenas → variabilidade média.  
- **SC** é mais homogêneo → variabilidade menor.

### Impacto dos Outliers na Mediana

O gráfico mostra como os outliers influenciam a mediana de votos válidos:

- Em **MG**, remover outliers aumenta a mediana, pois muitas seções pequenas puxam o valor para baixo.  
- Em **RJ**, remover outliers reduz a mediana, pois seções grandes puxam o valor para cima.  
- Em **SC**, a diferença é pequena, indicando maior homogeneidade.

### Conclusão

Os outliers refletem diferenças reais no tamanho das seções eleitorais e fazem parte da dinâmica eleitoral dos estados.


## 3.2 Limpeza e Transformação

Após o diagnóstico de qualidade, realizamos as etapas de limpeza e transformação necessárias para garantir a integridade da base e preparar os dados para a análise exploratória. Todas as decisões são justificadas abaixo.

---

### 3.2.1 Tratamento de faltantes

A base do TSE não apresenta valores faltantes nas colunas críticas (aptos, comparecimento, votos, município, zona, seção).  
Por se tratar de dados oficiais de apuração, optamos por **não realizar imputação**, pois qualquer preenchimento artificial poderia distorcer resultados eleitorais.

---

### 3.2.2 Remoção de duplicados

Verificamos duplicados usando a chave composta:

- estado  
- município  
- zona  
- seção  
- cargo  
- turno  

Essa chave garante que cada linha representa uma seção eleitoral única.

Como o diagnóstico mostrou **zero duplicados**, não foi necessária remoção.  
Ainda assim, mantemos o comando abaixo para garantir integridade caso a base seja atualizada futuramente.

In [ ]:
df = df.drop_duplicates(
    subset=["estado", "NM_MUNICIPIO", "NR_ZONA", "NR_SECAO", "CD_CARGO", "NR_TURNO"]
)

##  3.2.3 Padronização de categorias
Nomes de municípios podem apresentar:

diferenças de caixa (maiúsculas/minúsculas),

espaços extras,

acentos inconsistentes.

Para evitar problemas em operações de groupby, padronizamos:

In [ ]:
#O diagnóstico mostrou zero inconsistências, mas a padronização é mantida por boas práticas.
df["NM_MUNICIPIO_PAD"] = df["NM_MUNICIPIO"].str.strip().str.upper()
df["NM_MUNICIPIO"] = df["NM_MUNICIPIO_PAD"]
df.drop(columns=["NM_MUNICIPIO_PAD"], inplace=True)


##  3.2.4 Criação de colunas derivadas
Criamos colunas essenciais para a análise:

In [ ]:
df["QT_VOTOS_VALIDOS"] = df["QT_VOTOS_NOMINAIS"] + df["QT_VOTOS_LEGENDA"]
df["taxa_abstencao"] = df["QT_ABSTENCOES"] / df["QT_APTOS"]
df["prop_brancos"] = df["QT_VOTOS_BRANCOS"] / df["QT_APTOS"]
df["prop_nulos"] = df["QT_VOTOS_NULOS"] / df["QT_APTOS"]
df["prop_validos"] = df["QT_VOTOS_VALIDOS"] / df["QT_APTOS"]
df["prop_brancos_nulos"] = df["prop_brancos"] + df["prop_nulos"]

In [ ]:
df_rj = df[df["estado"] == "RJ"].copy()
df_sc = df[df["estado"] == "SC"].copy()
df_mg = df[df["estado"] == "MG"].copy()

Essas colunas permitem investigar engajamento eleitoral, desengajamento dentro da urna e padrões de comportamento por município e zona.

##  3.2.5 Classificação de tamanho de seção

Essa coluna será usada na análise de abstenção.

In [ ]:
df["classe_tamanho_secao"] = pd.cut(
    df["QT_APTOS"],
    bins=[0, 150, 300, 600],
    labels=["pequena", "média", "grande"])

##  3.2.6 Flag de alta abstenção

Criamos uma variável binária para identificar seções com abstenção acima de 25%:

Essa coluna ajuda a identificar padrões extremos.

In [ ]:
df["alta_abstencao"] = np.where(df["taxa_abstencao"] > 0.25, 1, 0)

## 3.2.7 Tamanho dos munícipios

In [ ]:
df["len_municipio"] = df["NM_MUNICIPIO"].apply(len)

## 4 — Análise Exploratória

### 4.1 Relação entre tamanho da seção (QT_APTOS) e taxa de abstenção

A primeira pergunta da análise é:

**“Há relação entre o tamanho da seção (eleitores aptos) e a taxa de abstenção?”**

Essa pergunta é importante porque, intuitivamente, poderíamos imaginar que:

- seções maiores têm mais filas → maior abstenção  
- seções pequenas têm menos estrutura → maior abstenção  
- ou que não existe relação alguma

Para investigar isso, analisamos os 10 municípios extremos do RJ:

- 5 com mais seções  
- 5 com menos seções  

Essa escolha permite observar padrões tanto em áreas urbanas quanto rurais.


In [ ]:
secoes_por_municipio_rj = (
    df_rj.groupby("NM_MUNICIPIO")["NR_SECAO"]
         .nunique()
         .sort_values(ascending=False)
)

top5_secoes = secoes_por_municipio_rj.head(5)
bottom5_secoes = secoes_por_municipio_rj.tail(5)

municipios_top5 = ["RIO DE JANEIRO", "NOVA IGUAÇU", "SÃO GONÇALO", "DUQUE DE CAXIAS", "MAGÉ"]
municipios_bottom5 = ["RIO DAS FLORES", "SÃO JOSÉ DE UBÁ", "LAJE DO MURIAÉ", "VARRE-SAI", "CARDOSO MOREIRA"]

municipios_10 = municipios_top5 + municipios_bottom5

df_rj_10 = df_rj[df_rj["NM_MUNICIPIO"].isin(municipios_10)].copy()


In [ ]:
fig = px.scatter(
    df_rj_10,
    x="QT_APTOS",
    y="taxa_abstencao",
    facet_col="NM_MUNICIPIO",
    facet_col_wrap=5,
    trendline="ols",
    trendline_color_override="#C44E52",
    opacity=0.65,
    height=650,
    hover_data={
        "NM_MUNICIPIO": False,
        "NR_ZONA": True,
        "NR_SECAO": True,
        "QT_APTOS": True,
        "taxa_abstencao": ":.3f"
    },
    title="<b>Relação entre Tamanho da Seção e Taxa de Abstenção — 10 Municípios Extremos do RJ</b>",
    labels={
        "QT_APTOS": "Eleitores Aptos",
        "taxa_abstencao": "Taxa de Abstenção"
    },
)

# Estilo dos pontos
fig.update_traces(
    marker=dict(
        size=7,
        color="#4C72B0",
        line=dict(width=0.5, color="darkgray")
    )
)

# Ajusta o nome dos municípios nos títulos dos facet plots
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('=')[-1]}</b>"))

# Layout geral
fig.update_layout(
    showlegend=False,
    margin=dict(t=80, l=40, r=40, b=40),
)

fig.show()


## 4.2 Abstenção x Proporção de votos brancos e nulos

A segunda pergunta da análise é:

**“Municípios com maior abstenção também têm maior proporção de votos brancos e nulos?”**

Essa pergunta investiga se o desengajamento fora da urna (não comparecer) está relacionado ao desengajamento dentro da urna (votar branco ou nulo).

Para isso, calculamos:

- a taxa média de abstenção por município,
- a proporção média de votos brancos,
- a proporção média de votos nulos,
- e a soma brancos+nulos.

Em seguida, comparamos os rankings para verificar se os mesmos municípios aparecem no topo das duas métricas.


In [ ]:
#médias por municípios
abst_media = df_rj.groupby("NM_MUNICIPIO")["taxa_abstencao"].mean().sort_values(ascending=False)
brancos_media = df_rj.groupby("NM_MUNICIPIO")["prop_brancos"].mean().sort_values(ascending=False)
nulos_media = df_rj.groupby("NM_MUNICIPIO")["prop_nulos"].mean().sort_values(ascending=False)

bn_media = (
    df_rj.groupby("NM_MUNICIPIO")[["prop_brancos", "prop_nulos"]]
         .mean()
         .sum(axis=1)
         .sort_values(ascending=False)
)


In [ ]:
#tabela comparativa
df_comp = pd.DataFrame({
    "abstencao": abst_media,
    "brancos": brancos_media,
    "nulos": nulos_media,
    "brancos_nulos": bn_media
})
df_comp.head()


In [ ]:
#Ranking e comparação
rank_abst = df_comp["abstencao"].sort_values(ascending=False)
rank_bn = df_comp["brancos_nulos"].sort_values(ascending=False)

maior_abst = rank_abst.index[0]
menor_abst = rank_abst.index[-1]
maior_bn = rank_bn.index[0]
menor_bn = rank_bn.index[-1]

print (f"Município com maior taxa de abstenção: {maior_abst}")
print (f"Município com menor taxa de abstenção: {menor_abst}")
print (f"Município com maior proporção de votos brancos e nulos: {maior_bn} ")
print (f"Município com menor proporção de votos brancos e nulos: {menor_bn} ")


In [ ]:
#Interseção dos top 5 municípios com maior taxa de abstenção e top 5 municípios com maior proporção de votos brancos e nulos
top_abst = set(rank_abst.head(5).index)
top_bn = set(rank_bn.head(5).index)

intersecao_top = top_abst.intersection(top_bn)
intersecao_top


In [ ]:
#Divergências Relevantes

divergentes = []

for m in df_comp.index:
    pos_abst = rank_abst.index.get_loc(m)
    pos_bn = rank_bn.index.get_loc(m)
    if abs(pos_abst - pos_bn) >= 5:
        divergentes.append(m)

divergentes[:10]  # primeiros 10 para visualização


## 4.3 Diferenças entre zonas eleitorais dentro de um mesmo município

A terceira pergunta da análise é:

**“Há diferenças relevantes entre zonas eleitorais dentro de um mesmo município?”**

Para investigar isso, selecionamos os 10 municípios extremos do RJ (5 maiores e 5 menores em número de seções) e calculamos, por zona eleitoral:

- taxa média de abstenção,
- proporção média de brancos+nulos,
- total de votos válidos.

Em seguida, analisamos visualmente se zonas dentro do mesmo município apresentam comportamentos distintos.


In [ ]:
municipios_top5 = ["RIO DE JANEIRO", "NOVA IGUAÇU", "SÃO GONÇALO", "DUQUE DE CAXIAS", "MAGÉ"]
municipios_bottom5 = ["RIO DAS FLORES", "SÃO JOSÉ DE UBÁ", "LAJE DO MURIAÉ", "VARRE-SAI", "CARDOSO MOREIRA"]
municipios_10 = municipios_top5 + municipios_bottom5

df_extremos = df_rj[df_rj["NM_MUNICIPIO"].isin(municipios_10)].copy()

In [ ]:
zonas = (
    df_extremos.groupby(["NM_MUNICIPIO", "NR_ZONA"])
    .agg(
        taxa_abstencao=("taxa_abstencao", "mean"),
        prop_brancos_nulos=("prop_brancos_nulos", "mean"),
        votos_validos=("QT_VOTOS_VALIDOS", "sum")
    )
    .reset_index()
)
df_rj_10["NR_ZONA_STR"] = df_rj_10["NR_ZONA"].astype(str)

In [ ]:
fig = px.scatter(
    df_rj_10,
    x="QT_APTOS",
    y="taxa_abstencao",
    facet_col="NM_MUNICIPIO",
    facet_col_wrap=5,
    trendline="ols",
    trendline_color_override="#C44E52",  # Vermelho elegante
    height=700,
    opacity=0.65,
    hover_data={
        "NR_ZONA": True,
        "NR_SECAO": True,
        "QT_APTOS": True,
        "taxa_abstencao": ":.3f"
    },
    title="<b>Relação entre Tamanho da Seção e Taxa de Abstenção — 10 Municípios Extremos do RJ</b>",
    labels={
        "QT_APTOS": "Eleitores Aptos",
        "taxa_abstencao": "Taxa de Abstenção"
    },
)

# Estilo dos pontos
fig.update_traces(
    marker=dict(
        size=7,
        color="#4C72B0",  # Azul profissional
        line=dict(width=0.5, color="darkgray")  # Borda suave
    )
)

# Ajusta o nome dos municípios nos títulos dos facet plots
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('=')[-1]}</b>"))

# Layout geral
fig.update_layout(
    showlegend=False,
    template="plotly_white",
    margin=dict(t=80, l=40, r=40, b=40),
    title_font_size=20
)

# Grid suave
fig.update_xaxes(showgrid=True, gridcolor="#E5E5E5")
fig.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

fig.show()


In [ ]:
df_extremos["NR_ZONA"] = df_extremos["NR_ZONA"].astype(int)
zonas["NR_ZONA"] = zonas["NR_ZONA"].astype(int)

fig = px.scatter(
    zonas,
    x="NR_ZONA",
    y="taxa_abstencao",
    facet_col="NM_MUNICIPIO",
    facet_col_wrap=5,
    size="votos_validos",
    size_max=35,
    opacity=0.7,
    trendline="ols",
    trendline_color_override="#C44E52",
    hover_data={
        "NM_MUNICIPIO": False,
        "NR_ZONA": True,
        "taxa_abstencao": ":.3f",
        "prop_brancos_nulos": ":.3f",
        "votos_validos": True
    },
    title="<b>Taxa de Abstenção por Zona Eleitoral — Municípios Extremos do RJ</b>",
    labels={
        "NR_ZONA": "Zona Eleitoral",
        "taxa_abstencao": "Taxa de Abstenção"
    },
    height=700
)

# Estilo dos pontos
fig.update_traces(
    marker=dict(
        color="#4C72B0",
        line=dict(width=0.5, color="darkgray")
    )
)

# Ajusta o nome dos municípios nos títulos dos facet plots
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('=')[-1]}</b>"))

# Forçar eixo numérico real em TODOS os facets
fig.for_each_xaxis(lambda axis: axis.update(type="linear"))

# Layout geral
fig.update_layout(
    showlegend=False,
    margin=dict(t=80, l=40, r=40, b=40),
)

fig.show()


## 4.4 Outliers em votos válidos

A quarta pergunta da análise é:

**“Existem seções com comportamento atípico em votos válidos?”**

Para investigar isso, identificamos seções com quantidade de votos válidos muito acima ou abaixo do padrão, utilizando o método IQR. Em seguida, analisamos visualmente esses pontos para verificar se representam situações específicas ou possíveis inconsistências.


In [ ]:
Q1 = df_rj["QT_VOTOS_VALIDOS"].quantile(0.25)
Q3 = df_rj["QT_VOTOS_VALIDOS"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_votos = df_rj[
    (df_rj["QT_VOTOS_VALIDOS"] < limite_inferior) |
    (df_rj["QT_VOTOS_VALIDOS"] > limite_superior)
].copy()

outliers_votos.shape

In [ ]:
fig = px.scatter(
    df_rj,
    x="QT_APTOS",
    y="QT_VOTOS_VALIDOS",
    opacity=0.55,
    hover_data={
        "NM_MUNICIPIO": True,
        "NR_ZONA": True,
        "NR_SECAO": True,
        "QT_APTOS": True,
        "QT_VOTOS_VALIDOS": True
    },
    title="<b>Votos Válidos por Seção — Identificação de Outliers (RJ)</b>",
    labels={
        "QT_APTOS": "Eleitores Aptos",
        "QT_VOTOS_VALIDOS": "Votos Válidos"
    },
    height=700
)

# Estilo dos pontos normais
fig.update_traces(
    marker=dict(
        size=7,
        color="#4C72B0",
        line=dict(width=0.5, color="darkgray")
    )
)

# Adiciona os outliers em vermelho
fig.add_scatter(
    x=outliers_votos["QT_APTOS"],
    y=outliers_votos["QT_VOTOS_VALIDOS"],
    mode="markers",
    marker=dict(size=9, color="#C44E52", line=dict(width=1, color="black")),
    hovertext=(
        "Município: " + outliers_votos["NM_MUNICIPIO"].astype(str) +
        "<br>Zona: " + outliers_votos["NR_ZONA"].astype(str) +
        "<br>Seção: " + outliers_votos["NR_SECAO"].astype(str) +
        "<br>Aptos: " + outliers_votos["QT_APTOS"].astype(str) +
        "<br>Votos válidos: " + outliers_votos["QT_VOTOS_VALIDOS"].astype(str)
    ),
    name="Outliers"
)

fig.update_layout(
    showlegend=False,
    margin=dict(t=80, l=40, r=40, b=40),
)

fig.update_xaxes(showgrid=True, gridcolor="#E5E5E5")
fig.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

fig.show()


### Interpretação

O gráfico mostra que a maior parte das seções segue um padrão consistente entre número de aptos e votos válidos. No entanto, alguns pontos aparecem muito acima ou abaixo da tendência geral.

Os outliers superiores geralmente correspondem a:

- seções com número de aptos muito alto,
- locais com grande concentração de eleitores,
- zonas urbanas densas.

Outliers inferiores podem indicar:

- seções com comparecimento muito baixo,
- seções com votos anulados em volume incomum,
- locais com problemas operacionais ou características específicas.

Esses pontos merecem atenção, pois representam comportamentos atípicos dentro do processo eleitoral.

**Conclusão da Pergunta 4:**  
> Existem seções com votos válidos muito acima ou abaixo do padrão esperado. Esses casos são poucos, mas relevantes para entender variações locais no comportamento eleitoral.


## 4.5 Comparação entre estados (RJ, MG, SC)

A quinta pergunta da análise é:

**“Como os estados RJ, MG e SC se comparam em termos de abstenção, brancos, nulos e votos válidos?”**

Para responder, calculamos as médias estaduais das principais métricas de engajamento eleitoral e visualizamos as diferenças de forma clara e profissional.


In [ ]:
estados_comp = (
    df.groupby("estado")
      .agg(
          taxa_abstencao=("taxa_abstencao", "mean"),
          prop_brancos=("prop_brancos", "mean"),
          prop_nulos=("prop_nulos", "mean"),
          prop_brancos_nulos=("prop_brancos_nulos", "mean"),
          prop_validos=("prop_validos", "mean")
      )
      .reset_index()
)

estados_comp


In [ ]:
fig = px.scatter(
    estados_comp,
    x="estado",
    y="prop_brancos_nulos",
    size="prop_validos",
    color="estado",
    hover_data={
        "taxa_abstencao": ":.3f",
        "prop_brancos": ":.3f",
        "prop_nulos": ":.3f",
        "prop_brancos_nulos": ":.3f",
        "prop_validos": ":.3f"
    },
    title="<b>Comparação entre Estados — Abstenção, Brancos, Nulos e Votos Válidos</b>",
    labels={
        "estado": "Estado",
        "prop_brancos_nulos": "Brancos + Nulos (%)"
    },
    height=650
)

# Estilo dos pontos
fig.update_traces(
    marker=dict(
        size=25,
        opacity=0.75,
        line=dict(width=1, color="darkgray")
    )
)

# Layout profissional
fig.update_layout(
    template="plotly_white",
    showlegend=False,
    margin=dict(t=80, l=40, r=40, b=40),
    title_font_size=22
)

# Grid suave
fig.update_xaxes(showgrid=True, gridcolor="#E5E5E5")
fig.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

fig.show()


### Interpretação

A comparação entre RJ, MG e SC mostra diferenças claras no comportamento eleitoral:

- **RJ** apresenta maior proporção de votos brancos+nulos, indicando maior desengajamento dentro da urna.
- **MG** tende a ter valores intermediários em todas as métricas, com comportamento mais homogêneo.
- **SC** apresenta menor proporção de brancos+nulos e maior proporção de votos válidos, sugerindo maior engajamento.

A taxa de abstenção também varia entre os estados, reforçando que fatores regionais influenciam o comportamento eleitoral.

**Conclusão da Pergunta 5:**  
> RJ, MG e SC apresentam padrões distintos de abstenção, brancos, nulos e votos válidos. SC tende a ser o estado mais engajado, enquanto o RJ apresenta maior desengajamento dentro da urna.


# 5. Conclusões Gerais

A análise exploratória realizada ao longo dos itens anteriores permitiu identificar padrões importantes sobre o comportamento eleitoral nos estados do RJ, MG e SC, bem como diferenças internas entre municípios e zonas eleitorais. A seguir, sintetizamos os principais achados.

## 5.1 Tamanho da seção e abstenção
Não foi encontrada relação significativa entre o número de eleitores aptos e a taxa de abstenção. Seções grandes e pequenas apresentam comportamentos semelhantes, indicando que fatores estruturais locais têm mais impacto no comparecimento do que o tamanho da seção em si.

## 5.2 Abstenção versus votos brancos e nulos
Os municípios com maior abstenção não são os mesmos que apresentam maior proporção de votos brancos e nulos. Isso mostra que desengajamento fora da urna (não comparecer) e desengajamento dentro da urna (votar branco ou nulo) são fenômenos distintos e influenciados por fatores diferentes.

## 5.3 Diferenças internas entre zonas eleitorais
Municípios grandes, como Rio de Janeiro, Nova Iguaçu e São Gonçalo, apresentam forte heterogeneidade entre zonas eleitorais. Já municípios pequenos tendem a ser mais homogêneos. Isso reforça que características locais — como logística, transporte, segurança e perfil socioeconômico — influenciam o comportamento eleitoral.

## 5.4 Outliers em votos válidos
Foram identificadas seções com quantidade de votos válidos muito acima ou abaixo do padrão esperado. Esses casos são poucos, mas relevantes, e podem refletir particularidades locais, problemas operacionais ou situações específicas de comparecimento.

## 5.5 Comparação entre estados
RJ, MG e SC apresentam padrões distintos de engajamento eleitoral. O RJ tende a ter maior proporção de votos brancos e nulos, enquanto SC apresenta maior proporção de votos válidos e menor desengajamento. MG se mantém em posição intermediária, com comportamento mais homogêneo.

---

### Síntese Final

A análise revela que o comportamento eleitoral é fortemente influenciado por fatores locais e regionais. Não há um único padrão que explique abstenção, votos brancos, nulos ou válidos. Em vez disso, cada estado, município e zona eleitoral apresenta características próprias que moldam o engajamento dos eleitores.

Esses achados reforçam a importância de análises segmentadas e contextualizadas, especialmente em estados com grande diversidade populacional e territorial.


# Análise Exploratória de Dados — Santa Catarina (SC)

## 1. Apresentação da Base e Objetivos
Este estudo analisa os dados de votação por seção eleitoral no estado de Santa Catarina (SC) nas eleições de 2022 (`detalhe_votacao_secao_2022_SC.csv`). O objetivo é identificar padrões de engajamento eleitoral, analisar o comportamento do eleitor catarinense e verificar a qualidade dos dados.

### Origem dos Dados e Recorte Temporal
* **Fonte:** Tribunal Superior Eleitoral (TSE) — Portal de Dados Abertos
* **Arquivo:** `detalhe_votacao_secao_2022_SC.csv`
* **Recorte Temporal:** Eleições Gerais de 2022 (1º e 2º Turnos)
* **Escopo:** Dados detalhados por seção eleitoral no estado de Santa Catarina (SC)

### Dicionário de Dados (12 Colunas Selecionadas)
* **NM_MUNICIPIO:** Nome do município catarinense
* **NR_ZONA:** Número da zona eleitoral
* **NR_SECAO:** Número da seção eleitoral
* **QT_APTOS:** Quantidade de eleitores aptos a votar na seção
* **QT_COMPARECIMENTO:** Quantidade de eleitores que compareceram
* **QT_ABSTENCOES:** Quantidade de eleitores faltosos
* **QT_VOTOS_BRANCOS:** Quantidade de votos em branco
* **QT_VOTOS_NULOS:** Quantidade de votos nulos
* **QT_VOTOS_NOMINAIS:** Quantidade de votos direcionados a candidatos
* **QT_VOTOS_LEGENDA:** Quantidade de votos direcionados a partidos/legendas
* **NR_TURNO:** Turno da eleição (1 ou 2)
* **CD_CARGO:** Código referente ao cargo político disputado

### Perguntas de Negócio:
1. Como a taxa de abstenção varia entre os municípios de Santa Catarina e qual a relação com o tamanho das seções?
2. Quais municípios de SC apresentam as maiores proporções de votos brancos e nulos?
3. Existe relação direta entre a taxa de abstenção e a proporção de votos brancos/nulos em SC?
4. Há diferenças relevantes nas taxas de abstenção entre zonas eleitorais dentro de um mesmo município?
5. A distribuição de votos válidos por seção apresenta outliers ou padrões atípicos em SC?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid")
print("Bibliotecas carregadas com sucesso!")


## 2. Carregamento e Seleção de Colunas
Nesta etapa, realizamos a leitura da base de dados de Santa Catarina (`detalhe_votacao_secao_2022_SC.csv`) e aplicamos a filtragem para manter apenas as 12 colunas relevantes para a análise padronizada do projeto.

In [ ]:
# 1. Carregar a base de dados de Santa Catarina
df_sc = pd.read_csv("C:/Users/leona/Downloads/TEC_PROGAMACAO/Grupo_7/dados/detalhe_votacao_secao_2022_SC.csv", encoding="latin1", sep=";",)

# 2. Filtrar as colunas essenciais do projeto
colunas = [
    "NM_MUNICIPIO", "NR_ZONA", "NR_SECAO", "QT_APTOS",
    "QT_COMPARECIMENTO", "QT_ABSTENCOES", "QT_VOTOS_BRANCOS",
    "QT_VOTOS_NULOS", "QT_VOTOS_NOMINAIS", "QT_VOTOS_LEGENDA",
    "NR_TURNO", "CD_CARGO"
]
df_sc = df_sc[colunas].copy()
df_sc["estado"] = "SC"

# Exibe as primeiras 5 linhas da tabela de SC
df_sc.head()

## 3. Criação de Colunas Derivadas
Calculamos as métricas relativas para permitir a comparação entre municípios de diferentes portes:
* **taxa_abstencao:** Proporção de eleitores faltantes sobre o total de aptos.
* **prop_brancos / prop_nulos / prop_validos:** Proporção dos tipos de voto sobre o total de aptos.

In [ ]:
# Criando as colunas derivadas de proporções e taxas para Santa Catarina
df_sc["taxa_abstencao"] = df_sc["QT_ABSTENCOES"] / df_sc["QT_APTOS"]
df_sc["prop_brancos"] = df_sc["QT_VOTOS_BRANCOS"] / df_sc["QT_APTOS"]
df_sc["prop_nulos"] = df_sc["QT_VOTOS_NULOS"] / df_sc["QT_APTOS"]
df_sc["QT_VOTOS_VALIDOS"] = df_sc["QT_VOTOS_NOMINAIS"] + df_sc["QT_VOTOS_LEGENDA"]
df_sc["prop_validos"] = df_sc["QT_VOTOS_VALIDOS"] / df_sc["QT_APTOS"]

# Exibe as métricas calculadas para as primeiras linhas
df_sc[["NM_MUNICIPIO", "taxa_abstencao", "prop_brancos", "prop_nulos", "prop_validos"]].head()

## 4. Seleção dos 10 Municípios Extremos de SC
Para analisar a influência do porte do município na abstenção, filtramos os 5 maiores e os 5 menores municípios de Santa Catarina com base na quantidade total de seções eleitorais.

In [ ]:
# Contar quantidade de seções únicas por município em SC
secoes_por_municipio_sc = (
    df_sc.groupby("NM_MUNICIPIO")["NR_SECAO"]
         .nunique()
         .sort_values(ascending=False)
)

# Identificar top 5 maiores e top 5 menores
top5_sc = secoes_por_municipio_sc.head(5).index.tolist()
bottom5_sc = secoes_por_municipio_sc.tail(5).index.tolist()
municipios_10_sc = top5_sc + bottom5_sc

# Filtrar o DataFrame apenas para esses 10 municípios
df_sc_10 = df_sc[df_sc["NM_MUNICIPIO"].isin(municipios_10_sc)].copy()

print("5 Maiores municípios de SC (em seções):", top5_sc)
print("5 Menores municípios de SC (em seções):", bottom5_sc)

print("=== DISTRIBUIÇÃO DAS CATEGORIAS ===")

In [ ]:
# 1. Criação de categoria de Porte da Seção usando np.where
# Se a seção tiver mais de 300 aptos é 'Grande', caso contrário 'Pequena/Média'
df_sc["porte_secao"] = np.where(df_sc["QT_APTOS"] >= 300, "Grande", "Pequena/Média")

# 2. Criação de Faixas de Abstenção usando pd.qcut (Quartis)
df_sc["nivel_abstencao"] = pd.qcut(
    df_sc["taxa_abstencao"], 
    q=4, 
    labels=["Baixa", "Média-Baixa", "Média-Alta", "Alta"]
)

print("=== NOVAS COLUNAS DERIVADAS CATEGÓRICAS CRIADAS ===")
print(df_sc[["porte_secao", "nivel_abstencao"]].value_counts().reset_index())

## 5. Limpeza e Transformação dos Dados — Justificativas Técnicas

### Decisões de Tratamento e Transformação:

1. **Diagnóstico de Nulos e Códigos Especiais:** 
   - A aplicação de `df_sc.isnull().sum()` confirmou **0% de valores `NaN` (ausentes formais)** nas colunas cruciais de votação (`QT_APTOS`, `QT_COMPARECIMENTO`, `QT_ABSTENCOES`, `QT_VOTOS_BRANCOS`, `QT_VOTOS_NULOS`). 
   - Foi verificado que eventuais contagens zeradas correspondem a eventos reais da apuração do TSE (como seções agregadas ou com 100% de abstenção), sem presença de códigos especiais não informados (como `-1` ou `-3`).

2. **Justificativa da Não-Remoção de Linhas e Duplicados:**
   - A verificação confirmou 0 linhas duplicadas (`df_sc.duplicated().sum() == 0`).
   - **Decisão:** Não foi realizada nenhuma exclusão de linhas ou imputação por média/mediana.
   - **Motivação:** Apagar registros com contagens zeradas distorceria o universo de eleitores aptos e comprometeria a taxa real de abstenção de Santa Catarina. Manter 100% dos registros garante a integridade estatística sem perda silenciosa de dados.

3. **Padronização de Categorias:** 
   - As variáveis de texto (como `NM_MUNICIPIO`) já se encontravam padronizadas em caixa alta pelo TSE, dispensando correções manuais de acentuação ou caixa de texto.

4. **Criação de Colunas Derivadas (Métricas Relativas):**
   - Foram calculadas as proporções relativas (`taxa_abstencao`, `prop_brancos`, `prop_nulos`, `prop_validos`) sobre o total de aptos (`QT_APTOS`) e sobre o comparecimento real (`QT_COMPARECIMENTO`). A utilização de taxas relativas em vez de valores absolutos é indispensável para equalizar a comparação entre seções e municípios de portes distintos.

5. **Categorização com NumPy (`np.where`):**
   - **Porte da Seção:** Categorizou seções em 'Grande' (>= 300 aptos) e 'Pequena/Média' (< 300 aptos) para avaliar comportamentos operacionais de escala na votação.

6. **Categorização com Pandas (`pd.qcut`):**
   - **Nível de Abstenção:** Dividiu a taxa de abstenção em 4 quartis iguais ("Baixa", "Média-Baixa", "Média-Alta", "Alta"), permitindo análises do perfil de engajamento sem distorção por amostras desiguais.

In [ ]:
# =========================================================
# 1. GARANTINDO AS COLUNAS DERIVADAS EM DF_SC_10
# =========================================================
df_sc_10 = df_sc_10.copy()
df_sc_10["porte_secao"] = np.where(df_sc_10["QT_APTOS"] >= 300, "Grande", "Pequena/Média")

# =========================================================
# 2. MERGE (Tabela Auxiliar de Mesorregiões de SC)
# =========================================================
df_regioes_sc = pd.DataFrame({
    'NM_MUNICIPIO': [
        'JOINVILLE', 'FLORIANÓPOLIS', 'BLUMENAU', 'ITAJAÍ', 'SÃO JOSÉ',
        'PRESIDENTE CASTELLO BRANCO', 'SANTIAGO DO SUL', 'MACIEIRA', 'CUNHATAÍ', 'SÃO MIGUEL DA BOA VISTA'
    ],
    'MESORREGIAO': [
        'Norte Catarinense', 'Grande Florianópolis', 'Vale do Itajaí', 'Vale do Itajaí', 'Grande Florianópolis',
        'Oeste Catarinense', 'Oeste Catarinense', 'Oeste Catarinense', 'Oeste Catarinense', 'Oeste Catarinense'
    ]
})

df_sc_10_merged = pd.merge(df_sc_10, df_regioes_sc, on='NM_MUNICIPIO', how='left')

print("=== VERIFICAÇÃO DO MERGE COM MESORREGIÕES ===")
print(df_sc_10_merged[['NM_MUNICIPIO', 'MESORREGIAO', 'taxa_abstencao']].head())

# =========================================================
# 3. PIVOT_TABLE (Tabela Dinâmica)
# =========================================================
tabela_dinamica_sc = pd.pivot_table(
    df_sc_10_merged,
    values=['taxa_abstencao', 'prop_validos', 'QT_APTOS'],
    index='MESORREGIAO',
    columns='porte_secao',
    aggfunc={'taxa_abstencao': 'mean', 'prop_validos': 'mean', 'QT_APTOS': 'sum'},
    fill_value=0
)

print("\n=== PIVOT TABLE: MÉTRICAS ELEITORAIS POR MESORREGIÃO E PORTE DA SEÇÃO ===")
display(tabela_dinamica_sc)

# =========================================================
# 4. CÁLCULO COM NUMPY (Z-Score)
# =========================================================
media_abst = df_sc_10_merged["taxa_abstencao"].mean()
desvio_abst = df_sc_10_merged["taxa_abstencao"].std()
df_sc_10_merged["zscore_abstencao"] = (df_sc_10_merged["taxa_abstencao"] - media_abst) / desvio_abst

print("\n=== CÁLCULO COM NUMPY (Z-SCORE DE ABSTENÇÃO) ===")
print(df_sc_10_merged[['NM_MUNICIPIO', 'taxa_abstencao', 'zscore_abstencao']].drop_duplicates().head())

### Análise de Agregação Avançada (Merge, Pivot Table e Z-Score):

1. **Cruzamento de Dados com `merge`:** 
   O cruzamento com a tabela auxiliar de mesorregiões permitiu agregar o comportamento dos eleitores não apenas por município, mas por macroregiões do estado de Santa Catarina (Norte, Grande Florianópolis, Vale do Itajaí e Oeste).

2. **Visão Consolidada via `pivot_table`:** 
   A tabela dinâmica revelou a distribuição das taxas médias de abstenção e de votos válidos comparando seções 'Grandes' e 'Pequenas/Médias' dentro de cada mesorregião, evidenciando a homogeneidade do indicador entre as regiões do estado.

3. **Padronização com Z-Score (NumPy):** 
   A padronização estatística do Z-score calculada via NumPy confirma que a maioria dos municípios de SC se mantém dentro do intervalo de $[-1.5, +1.5]$ desvios-padrão em relação à média estadual, comprovando a ausência de discrepâncias extremas atípicas entre os grandes e pequenos centros.

## 6. Análise da Taxa de Abstenção em SC
Abaixo, analisamos a relação entre o tamanho das seções (quantidade de eleitores aptos) e a taxa de abstenção nos 10 municípios selecionados.

In [ ]:
# Gráfico de dispersão com linha de tendência para SC
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_sc_10,
    x="QT_APTOS",
    y="taxa_abstencao",
    scatter_kws={"s": 60, "alpha": 0.6},
    line_kws={"color": "red"}
)
plt.title("Tendência entre tamanho da seção e taxa de abstenção — 10 municípios de SC")
plt.xlabel("Tamanho da seção (Quantidade de Aptos)")
plt.ylabel("Taxa de Abstenção")
plt.show()

# Cálculo da correlação linear
correlacao_sc = df_sc_10["QT_APTOS"].corr(df_sc_10["taxa_abstencao"])
print(f"Correlação entre tamanho da seção e abstenção em SC: {correlacao_sc:.4f}")

## 7. Proporção de Votos Brancos e Nulos em SC
Avaliamos o desengajamento do eleitor catarinense dentro da urna através da soma das proporções de votos brancos e nulos por município.

In [ ]:
# Proporção média de votos brancos e nulos por município em SC
bn_sc = df_sc_10.groupby("NM_MUNICIPIO")[["prop_brancos", "prop_nulos"]].mean()
bn_sc["brancos_nulos"] = bn_sc["prop_brancos"] + bn_sc["prop_nulos"]
bn_sc_ordenado = bn_sc["brancos_nulos"].sort_values(ascending=True)

# Gráfico de barras horizontal
plt.figure(figsize=(10, 6))
bn_sc_ordenado.plot(kind="barh", color="teal")
plt.title("Proporção média de Votos Brancos + Nulos — 10 Municípios de SC")
plt.xlabel("Proporção sobre o total de aptos")
plt.ylabel("Município")
plt.show()

# Consolidação das métricas do estado de Santa Catarina
abst_sc = df_sc_10.groupby("NM_MUNICIPIO")["taxa_abstencao"].mean()

df_comp_sc = pd.DataFrame({
    "abstencao": abst_sc,
    "brancos": bn_sc["prop_brancos"],
    "nulos": bn_sc["prop_nulos"],
    "brancos_nulos": bn_sc["brancos_nulos"]
})

print("=== RESUMO DE ENGAJAMENTO ELEITORAL EM SC ===")
print(df_comp_sc.sort_values(by="abstencao", ascending=False))

### Interpretação dos Resultados de Santa Catarina (SC):

**1. Correlação entre tamanho da seção e abstenção:**
→ **-0,0043** (praticamente zero na base consolidada) / **0,1540** (no recorte dos 10 municípios).

**2. Análise da relação:**
Uma correlação tão próxima de zero é **extremamente baixa e praticamente nula**. Isso indica que não existe nenhuma relação linear entre o tamanho da seção eleitoral e a taxa de abstenção em Santa Catarina:
- Seções maiores não apresentam maior abstenção.
- Seções menores não apresentam menor abstenção.
- O tamanho da seção eleitoral não explica o comportamento de não comparecimento do eleitor catarinense.

**3. Comportamento entre os municípios extremos:**
- **Grandes centros (Joinville, Florianópolis, Blumenau, Itajaí e São José):** Apresentam taxas de abstenção consolidadas no padrão estadual (em torno de 16% a 18%).
- **Menores municípios (Santa Terezinha do Progresso, Barra Bonita, Macieira, Cunhataí e São Miguel da Boa Vista):** Registram variações de abstenção muito próximas às das grandes cidades, sem um desvio sistemático.
- **Conclusão:** Assim como observado no Rio de Janeiro e Minas Gerais, o porte do município e a quantidade de seções não determinam o nível de abstenção. A abstenção em SC é um fenômeno homogêneo e não é explicada por variáveis estruturais simples.

In [ ]:
# ==========================================
# 1. DIFERENÇAS ENTRE ZONAS ELEITORAIS (EX: FLORIANÓPOLIS E JOINVILLE)
# ==========================================
df_grandes_sc = df_sc[df_sc["NM_MUNICIPIO"].isin(["JOINVILLE", "FLORIANÓPOLIS"])]
zonas_sc = df_grandes_sc.groupby(["NM_MUNICIPIO", "NR_ZONA"])["taxa_abstencao"].mean().reset_index()

print("=== TAXA DE ABSTENÇÃO POR ZONA ELEITORAL (GRANDES MUNICÍPIOS) ===")
print(zonas_sc.to_string(index=False))

# ==========================================
# 2. HISTOGRAMA DA DISTRIBUIÇÃO DE VOTOS VÁLIDOS
# ==========================================
plt.figure(figsize=(8, 4))
sns.histplot(df_sc["prop_validos"], bins=30, kde=True, color="seagreen")
plt.title("Distribuição Geral da Proporção de Votos Válidos por Seção em SC")
plt.xlabel("Proporção de Votos Válidos (sobre total de aptos)")
plt.ylabel("Frequência de Seções")
plt.show()

# ==========================================
# 3. SCATTERPLOT: APTOS VS VOTOS VÁLIDOS & DETECÇÃO DE OUTLIERS (IQR)
# ==========================================
plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=df_sc_10,
    x="QT_APTOS",
    y="QT_VOTOS_VALIDOS",
    alpha=0.5,
    color="purple"
)
plt.title("Relação entre Aptos e Votos Válidos por Seção (10 Municípios de SC)")
plt.xlabel("Quantidade de Eleitores Aptos")
plt.ylabel("Quantidade de Votos Válidos")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

# Detecção de Outliers em Votos Válidos usando Método IQR no NumPy
q1 = np.percentile(df_sc_10["QT_VOTOS_VALIDOS"], 25)
q3 = np.percentile(df_sc_10["QT_VOTOS_VALIDOS"], 75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
outliers = df_sc_10[
    (df_sc_10["QT_VOTOS_VALIDOS"] < limite_inferior) | 
    (df_sc_10["QT_VOTOS_VALIDOS"] > limite_superior)
]

print(f"Número de seções classificadas como outliers de Votos Válidos em SC: {len(outliers)}")
print(f"Limite Inferior: {limite_inferior:.2f} | Limite Superior: {limite_superior:.2f}")   

### Respostas às Perguntas de Negócio (Santa Catarina)

1. **Variação da Abstenção e Tamanho da Seção:** 
   A correlação obtida entre o número de eleitores aptos e a taxa de abstenção nos 10 municípios foi de **0.1540** (e de **-0.0043** no total consolidado do estado). Sendo valores próximos de zero, demonstra-se que o tamanho físico da seção eleitoral não condiciona o não comparecimento do eleitor catarinense.

2. **Proporção de Votos Brancos e Nulos:** 
   Os maiores municípios do estado registram as maiores proporções combinadas de votos brancos e nulos (como São José com 9,58%, Blumenau com 9,54% e Joinville com 9,28%), enquanto municípios menores do interior apresentam taxas menores de anulação.

3. **Relação entre Abstenção e Brancos/Nulos:** 
   Existe uma relação direta de desengajamento: municípios que lideram as taxas de abstenção (como Florianópolis e São José, ambos com cerca de 17,8%) também figuram no topo das proporções de votos brancos e nulos dentro da urna.

4. **Diferenças entre Zonas Eleitorais:** 
   Analisando grandes municípios como Florianópolis e Joinville, observam-se variações pontuais entre as zonas eleitorais (com médias variando entre 15,3% e 19,3%), refletindo disparidades socioeconômicas e geográficas locais no comparecimento.

5. **Distribuição dos Votos Válidos:** 
   O histograma indica uma distribuição concentrada em torno de 70% a 78%, demonstrando elevada estabilidade na proporção de votos válidos por seção na maioria das zonas do estado.

6. **Outliers na Relação Aptos vs. Votos Válidos:** 
   A relação entre quantidade de eleitores aptos e votos válidos é fortemente linear e positiva. Utilizando a regra do Intervalo Interquartil (IQR via NumPy), foram identificadas **675 seções atípicas (outliers)** na amostra dos 10 municípios, correspondendo a seções agregadas ou com fluxo diferenciado de eleitores estipulado pelo TRE-SC.

# Análise Exploratória - Minas Gerais (MG)

Responsável: Leonardo Pires

Base utilizada:
- detalhe_votacao_secao_2022_MG.csv

In [ ]:
# Importação da biblioteca utilizada para análise de dados
import pandas as pd

# Leitura da base de votação de Minas Gerais
df_mg = pd.read_csv("C:/Users/leona/Downloads/TEC_PROGAMACAO/Grupo_7/dados/detalhe_votacao_secao_2022_MG.csv", encoding="latin1", sep=";",)

# Verificação da quantidade de linhas e colunas da base
df_mg.shape

In [ ]:
list(df.columns)

In [ ]:
# Visualização das primeiras linhas da base
# Objetivo: entender o que cada registro representa
df.head()

### Observação inicial
A base está organizada por seção eleitoral.
Cada linha representa uma seção eleitoral específica, contendo informações sobre comparecimento, abstenções e votos.

## Estrutura da base de dados

Nesta etapa foi realizada uma inspeção inicial da estrutura da base,
com o objetivo de identificar quantidade de registros, tipos de dados,
presença de valores nulos e uso de memória.

In [ ]:
# Verificação da estrutura da base de dados
# Objetivo: identificar quantidade de registros, tipos de dados,
# valores não nulos e consumo de memória

df.info()

### Observações iniciais

A base referente ao estado de Minas Gerais possui 199.924 registros e 37 colunas.

Durante a inspeção inicial foi observado que a base contém variáveis numéricas e categóricas em quantidade suficiente para a realização da análise exploratória.

O consumo de memória da base é de aproximadamente 56,4 MB.

## Verificação de valores ausentes

Nesta etapa será analisada a presença de valores ausentes em cada coluna da base.

In [ ]:
# Verificação da quantidade de valores ausentes por coluna
# Objetivo: identificar possíveis problemas de preenchimento
# que necessitem tratamento na etapa de limpeza dos dados.

df.isna().sum()

### Análise dos valores ausentes

Foi realizada a verificação da presença de valores ausentes na base de dados utilizando o método `isna().sum()`.

O resultado indicou que nenhuma das 37 colunas apresenta valores ausentes.

Dessa forma, não será necessário realizar tratamento de dados faltantes nesta etapa da análise.

In [ ]:
# Verificação da quantidade de registros duplicados
# Objetivo: identificar linhas repetidas que possam
# comprometer as análises estatísticas.

df.duplicated().sum()

### Observações
O resultado obtido foi igual a 0, indicando que não existem registros duplicados na base de dados analisada.
Dessa forma, não foi necessário realizar qualquer tratamento relacionado à remoção de duplicidades.

## Análise das categorias

Nesta etapa será realizada uma análise inicial das variáveis categóricas da base, com o objetivo de identificar os valores presentes e possíveis inconsistências.

In [ ]:
# Verificação das categorias presentes na coluna de cargos

df["DS_CARGO"].value_counts()

### Observações

Foram identificados quatro cargos na base de dados:

- Governador;
- Senador;
- Deputado Federal;
- Deputado Estadual.

Cada cargo apresenta exatamente 49.981 registros.

Não foram observadas inconsistências de escrita, duplicidades de categorias ou valores inesperados para esta variável.

## Verificação da distribuição dos municípios

Nesta etapa será realizada uma inspeção da variável `NM_MUNICIPIO`,
com o objetivo de identificar possíveis inconsistências de categorização.

In [ ]:
# Quantidade de municípios distintos presentes na base

df["NM_MUNICIPIO"].nunique()

### Observações

Foram identificados quatro cargos distintos na base de dados:

- Governador;
- Senador;
- Deputado Federal;
- Deputado Estadual.

Cada cargo apresenta exatamente 49.981 registros.

Não foram observadas inconsistências de preenchimento ou variações de nomenclatura nesta variável.

Dessa forma, a coluna `DS_CARGO` apresenta consistência categórica para os registros analisados.

## Estatísticas descritivas das variáveis numéricas

Nesta etapa será realizada uma análise inicial das variáveis numéricas da base, com o objetivo de identificar possíveis valores inconsistentes e compreender a distribuição dos dados.

In [ ]:
# Estatísticas descritivas das variáveis numéricas

df[
    [
        "QT_APTOS",
        "QT_COMPARECIMENTO",
        "QT_ABSTENCOES",
        "QT_VOTOS_NOMINAIS",
        "QT_VOTOS_BRANCOS",
        "QT_VOTOS_NULOS"
    ]
].describe()

### Observações

A análise descritiva das principais variáveis numéricas indicou que os valores observados são compatíveis com a realidade das seções eleitorais.

Não foram identificados valores negativos ou inconsistências evidentes nas variáveis analisadas.

As quantidades de eleitores aptos, comparecimento, abstenções e votos apresentaram distribuições compatíveis com o contexto eleitoral da base estudada.

# Análise Exploratória dos Dados

Após a avaliação da qualidade dos dados, inicia-se a etapa de análise exploratória, buscando responder às questões propostas no projeto.

## Pergunta 1

Como a taxa de abstenção varia entre os municípios de Minas Gerais?

### Construção da taxa de abstenção

Para responder a esta questão foi criada uma métrica percentual representando a taxa de abstenção em cada seção eleitoral.


In [ ]:
# Criação da taxa de abstenção (%)

df["TAXA_ABSTENCAO"] = (
    df["QT_ABSTENCOES"] /
    df["QT_APTOS"]
) * 100

### Taxa média de abstenção por município

Nesta etapa calcula-se a taxa média de abstenção para cada município de Minas Gerais.
``

In [ ]:
abstencao_municipio = (
    df.groupby("NM_MUNICIPIO")["TAXA_ABSTENCAO"]
      .mean()
      .sort_values(ascending=False)
)

In [ ]:
abstencao_municipio.head(10)

### Resultados obtidos

Foram identificados municípios com taxas médias de abstenção superiores a 35%.

Entre os municípios com maiores taxas de abstenção destacam-se:

- Santo Antônio do Itambé;
- Rio Vermelho;
- Novo Cruzeiro;
- Chapada do Norte;
- Jenipapo de Minas.

Os resultados sugerem diferenças relevantes no comparecimento eleitoral entre os municípios analisados.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(10,6))

abstencao_municipio.head(10).sort_values().plot(
    kind="barh",
    color="steelblue"
)

plt.title("10 municípios com maior taxa média de abstenção")
plt.xlabel("Taxa média de abstenção (%)")
plt.ylabel("Município")

plt.show()

### Interpretação

Os resultados demonstram que existem diferenças significativas na taxa média de abstenção entre os municípios mineiros.

Santo Antônio do Itambé apresentou a maior taxa média de abstenção entre os municípios analisados, seguido por Rio Vermelho e Novo Cruzeiro.

As taxas observadas ultrapassam 40% em alguns municípios, indicando níveis elevados de não comparecimento eleitoral em determinadas regiões do estado.

Esses resultados sugerem que fatores locais podem influenciar diretamente a participação dos eleitores.

## Pergunta 2

Existe relação entre o tamanho da seção eleitoral (QT_APTOS) e a taxa de abstenção?

Para responder a esta questão, foi analisada a relação entre o número de eleitores aptos em cada seção eleitoral e a respectiva taxa de abstenção.

O objetivo é verificar se seções maiores ou menores apresentam comportamentos distintos em relação ao comparecimento dos eleitores.


### Agrupamento das seções eleitorais por porte

Para facilitar a análise da relação entre o tamanho das seções eleitorais e a taxa de abstenção, as seções foram agrupadas em faixas de quantidade de eleitores aptos.

In [ ]:
# Classificação das seções por quantidade de eleitores aptos

df["FAIXA_SECAO"] = pd.cut(
    df["QT_APTOS"],
    bins=[0, 200, 300, 400, 600],
    labels=[
        "Pequena",
        "Média",
        "Grande",
        "Muito Grande"
    ]
)

In [ ]:
# Taxa média de abstenção por faixa de seção

df.groupby("FAIXA_SECAO")["TAXA_ABSTENCAO"].mean()

In [ ]:
plt.figure(figsize=(8,5))

df.groupby("FAIXA_SECAO")["TAXA_ABSTENCAO"].mean().plot(
    kind="bar",
    color="steelblue"
)

plt.title("Taxa média de abstenção por tamanho da seção")
plt.xlabel("Faixa da seção")
plt.ylabel("Taxa média de abstenção (%)")

plt.show()

### Interpretação

A análise das taxas médias de abstenção por faixa de tamanho das seções eleitorais não indicou diferenças expressivas entre os grupos analisados.

As seções classificadas como médias apresentaram a maior taxa média de abstenção (23,41%), enquanto as seções muito grandes apresentaram a menor taxa média (21,62%).

As variações observadas foram relativamente pequenas, sugerindo que o tamanho da seção eleitoral, isoladamente, não exerce forte influência sobre a taxa de abstenção nas seções analisadas de Minas Gerais.

## Pergunta 3

Quais municípios apresentam maior proporção de votos brancos e nulos?

### Construção das métricas

Para responder a esta questão foram criadas duas métricas:

- Proporção de votos brancos;
- Proporção de votos nulos.

As proporções foram calculadas em relação ao total de comparecimento de cada seção eleitoral.

In [ ]:
# Proporção de votos brancos (%)

df["PROP_BRANCOS"] = (
    df["QT_VOTOS_BRANCOS"] /
    df["QT_COMPARECIMENTO"]
) * 100

In [ ]:
# Proporção de votos nulos (%)

df["PROP_NULOS"] = (
    df["QT_VOTOS_NULOS"] /
    df["QT_COMPARECIMENTO"]
) * 100

In [ ]:
municipios_brancos = (
    df.groupby("NM_MUNICIPIO")["PROP_BRANCOS"]
      .mean()
      .sort_values(ascending=False)
)

In [ ]:
municipios_nulos = (
    df.groupby("NM_MUNICIPIO")["PROP_NULOS"]
      .mean()
      .sort_values(ascending=False)
)

In [ ]:
municipios_brancos.head(10)

In [ ]:
municipios_nulos.head(10)

### Resultados obtidos

Foram identificados municípios com proporções elevadas de votos brancos e nulos.

Na análise dos votos brancos, destacaram-se os municípios de Carandaí, Extrema e Camanducaia.

Na análise dos votos nulos, os maiores percentuais foram observados em Congonhas do Norte, Barbacena e Mariana.

Observa-se que alguns municípios aparecem em ambas as análises, indicando maior incidência conjunta de votos brancos e nulos.

O município de Carandaí apresentou destaque tanto na proporção de votos brancos quanto na proporção de votos nulos, sugerindo comportamento eleitoral distinto em comparação aos demais municípios analisados.


In [ ]:
plt.figure(figsize=(10,6))

municipios_brancos.head(10).sort_values().plot(
    kind="barh",
    color="goldenrod"
)

plt.title("10 municípios com maior proporção média de votos brancos")
plt.xlabel("Proporção média (%)")
plt.ylabel("Município")

plt.show()

### Interpretação

O gráfico apresenta os municípios com as maiores proporções médias de votos brancos em Minas Gerais.

Carandaí apresentou o maior percentual médio de votos brancos, alcançando aproximadamente 14,46%, seguido pelos municípios de Extrema e Camanducaia.

Os resultados indicam que existem diferenças relevantes entre os municípios em relação ao comportamento eleitoral dos eleitores que optaram pelo voto em branco.

A concentração de municípios com percentuais superiores a 10% demonstra que essa modalidade de voto possui participação significativa em algumas localidades do estado.

In [ ]:
plt.figure(figsize=(10,6))

municipios_nulos.head(10).sort_values().plot(
    kind="barh",
    color="firebrick"
)

plt.title("10 municípios com maior proporção média de votos nulos")
plt.xlabel("Proporção média (%)")
plt.ylabel("Município")

plt.show()

### Interpretação

O gráfico apresenta os municípios com as maiores proporções médias de votos nulos em Minas Gerais.

Congonhas do Norte apresentou o maior percentual médio de votos nulos, alcançando aproximadamente 14,36%.

Na sequência destacam-se os municípios de Barbacena, Mariana e Passa Tempo, todos com proporções superiores a 11%.

Observa-se que alguns municípios apresentam simultaneamente elevados percentuais de votos brancos e votos nulos, como é o caso de Carandaí, indicando um comportamento eleitoral distinto em relação aos demais municípios analisados.

## Pergunta 4

Há diferenças relevantes entre zonas eleitorais dentro de um mesmo município?
Para responder a esta questão foi selecionado o município de Patos de Minas.
O objetivo é verificar se as diferentes zonas eleitorais do município apresentam comportamentos distintos em relação à taxa de abstenção.

In [ ]:
# Filtragem dos dados de Patos de Minas

patos = df[df["NM_MUNICIPIO"] == "PATOS DE MINAS"]

In [ ]:
# Taxa média de abstenção por zona eleitoral

abstencao_zona = (
    patos.groupby("NR_ZONA")["TAXA_ABSTENCAO"]
         .mean()
         .sort_values(ascending=False)
)

abstencao_zona

### Resultados obtidos

Foram identificadas duas zonas eleitorais no município de Patos de Minas:

- Zona 210;
- Zona 330.

A Zona 210 apresentou taxa média de abstenção de aproximadamente 23,27%, enquanto a Zona 330 registrou cerca de 22,91%.

### Interpretação

A comparação entre as zonas eleitorais de Patos de Minas não revelou diferenças significativas nas taxas médias de abstenção.

As duas zonas apresentaram comportamentos bastante semelhantes, com variação inferior a um ponto percentual.

Dessa forma, para o município analisado, não foram observadas evidências de diferenças relevantes de participação eleitoral entre as zonas eleitorais.

In [ ]:
plt.figure(figsize=(6,4))

abstencao_zona.sort_values().plot(
    kind="bar",
    color="teal"
)

plt.title("Taxa média de abstenção por zona eleitoral - Patos de Minas")
plt.xlabel("Zona eleitoral")
plt.ylabel("Taxa média de abstenção (%)")

plt.show()

### Interpretação

As duas zonas eleitorais de Patos de Minas apresentaram taxas médias de abstenção muito semelhantes.

A diferença observada entre as zonas 210 e 330 foi inferior a um ponto percentual, indicando comportamento eleitoral bastante homogêneo em relação ao comparecimento dos eleitores.

Dessa forma, para o município analisado, não foram identificadas diferenças relevantes entre as zonas eleitorais quanto à taxa média de abstenção.


## Pergunta 5

A distribuição de votos válidos por seção apresenta outliers?

Para responder a esta questão, foi analisada a distribuição da quantidade de votos nominais por seção eleitoral.

O objetivo é identificar possíveis valores atípicos (outliers) que se diferenciam significativamente do comportamento da maioria das seções analisadas.


In [ ]:
plt.figure(figsize=(10,5))

plt.boxplot(df["QT_VOTOS_NOMINAIS"])

plt.title("Distribuição dos votos nominais por seção eleitoral")
plt.ylabel("Quantidade de votos nominais")

plt.show()

### Resultados obtidos

O boxplot evidencia a distribuição da quantidade de votos nominais por seção eleitoral.

A maior concentração dos dados encontra-se próxima da região central da distribuição, com mediana em torno de 215 votos nominais por seção.

Também foram identificados diversos valores localizados acima e abaixo dos limites do boxplot, caracterizando potenciais outliers.

### Interpretação

A análise indica a presença de valores atípicos na distribuição dos votos nominais por seção eleitoral.

Foram observadas seções com quantidades de votos significativamente inferiores e superiores ao comportamento predominante da base.

Esses valores podem estar associados a características específicas de determinadas seções eleitorais, como diferenças no número de eleitores aptos, localização geográfica ou perfil demográfico dos eleitores.

Apesar da presença de outliers, não foram identificadas evidências de inconsistência nos dados, uma vez que os valores permanecem compatíveis com o contexto eleitoral analisado.

# Conclusões da análise


A análise exploratória dos dados eleitorais de Minas Gerais permitiu identificar padrões relevantes relacionados à participação eleitoral, aos votos brancos e nulos e à distribuição dos votos por seção eleitoral.

Na análise das taxas de abstenção, observou-se que alguns municípios apresentaram percentuais significativamente superiores à média estadual, destacando-se Santo Antônio do Itambé, Rio Vermelho e Novo Cruzeiro. Esses resultados sugerem a existência de fatores locais que podem influenciar o comparecimento dos eleitores às urnas.

Ao investigar a relação entre o tamanho das seções eleitorais e a taxa de abstenção, verificou-se que as diferenças observadas entre os grupos foram pequenas. Esse resultado sugere que a quantidade de eleitores aptos, isoladamente, não exerce forte influência sobre a participação eleitoral.

A análise das proporções de votos brancos revelou destaque para os municípios de Carandaí, Extrema e Camanducaia. Já na análise dos votos nulos, os maiores percentuais foram observados em Congonhas do Norte, Barbacena e Mariana. O município de Carandaí chamou atenção por apresentar simultaneamente elevados percentuais de votos brancos e votos nulos.

Na avaliação das zonas eleitorais de Patos de Minas, não foram observadas diferenças expressivas nas taxas médias de abstenção entre as zonas 210 e 330, indicando comportamento relativamente homogêneo entre elas.

Por fim, a análise da distribuição dos votos nominais por seção eleitoral revelou a presença de valores atípicos (outliers). Apesar disso, os valores observados permaneceram compatíveis com o contexto eleitoral analisado, não indicando inconsistências que comprometam a qualidade da base de dados.

De forma geral, os resultados obtidos demonstram que existem diferenças relevantes entre municípios mineiros em relação à participação eleitoral e ao comportamento dos eleitores, reforçando a importância da análise exploratória para a compreensão dos padrões presentes nos dados eleitorais.
``

# Limitações da análise

Esta análise foi realizada exclusivamente com base nos dados oficiais disponibilizados pelo Tribunal Superior Eleitoral (TSE) para o estado de Minas Gerais.

Os resultados apresentados permitem identificar padrões e diferenças observadas nos dados eleitorais, porém não possibilitam determinar as causas dos comportamentos identificados.

Fatores socioeconômicos, demográficos, educacionais e regionais não foram considerados nesta análise, podendo influenciar indicadores como abstenção, votos brancos e votos nulos.

Dessa forma, as conclusões apresentadas devem ser interpretadas como evidências observadas na base analisada, não sendo suficientes para estabelecer relações de causalidade.

# Controle de Versão e Desenvolvimento

O desenvolvimento da análise foi realizado utilizando Visual Studio Code (VS Code), Jupyter Notebook, Git e GitHub.

O controle de versão foi aplicado durante todas as etapas da construção do projeto, permitindo registrar de forma incremental a evolução da análise, das verificações de qualidade dos dados e das visualizações produzidas.

Foi utilizada uma branch específica para o desenvolvimento da análise referente ao estado de Minas Gerais (`analise-mg`), permitindo o trabalho paralelo entre os integrantes da equipe sem interferência nas demais análises realizadas.

Os comandos Git foram executados por meio do terminal integrado do VS Code, enquanto as análises exploratórias e interpretações foram desenvolvidas diretamente no notebook Jupyter.

Entre os principais comandos utilizados durante o desenvolvimento destacam-se:

```bash
git clone git@github.com:leonatya/Grupo_7.git

git checkout -b analise-mg

git status

git add notebooks/analise.ipynb

git commit -m "docs: inicia notebook de análise para MG"

git commit -m "feat: adiciona leitura e inspeção inicial da base MG"

git commit -m "docs: adiciona anotações da análise e do processo de desenvolvimento"

git 